# Person 4 (Neha): correction model (code skeleton)
Plan: `correction_model_plan` (DRAFT until all four sign). **BERT router only.**

* **Section A (library) and Section B (dry run) are safe to run.** The dry run uses only the 80 pilot pairs and never loads the 152 confirmatory scores. Its numbers are **test output, not results**.
* **Section C (final run) is locked** behind `PLAN_SIGNED` and runs once, on the third set and then the 152 pairs.
* Everything here is **exploratory** (not in the frozen audit plan). Never overwrite a saved file: new versions get new names.

In [ ]:
from google.colab import files
up = files.upload()   # upload: scores_v2.csv, features_v2.csv, benchmark_v2.xlsx, config_v4.json

## A. Library (methods M0 to M4, measures, cross-validation)

In [ ]:
from IPython.display import display
import json, hashlib, itertools
import numpy as np, pandas as pd
from scipy import stats
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import cohen_kappa_score, roc_auc_score

SEED = 0
FEATS = ['hi_score', 'hi_tokens', 'hi_words', 'hi_fertility']      # allowed inputs (Hinglish only)
CUTS = {'10%': 0.5949, '20%': 0.5094, '30%': 0.4651, '50%': 0.4066, '70%': 0.3372}   # from config_v4.json

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()

def compute_features(df, tok, text_col='hinglish'):
    """Same recipe as Person02 / features_confirmatory_v1.csv: words = len(text.split()),
    tokens = len(router tokenizer tokens), fertility = tokens / words. NFC-normalised, stripped text."""
    import unicodedata
    texts = [unicodedata.normalize('NFC', str(x).strip()) for x in df[text_col]]
    f = pd.DataFrame({'pair_id': df['pair_id'].values})
    f[f'{text_col}_words'] = [len(t.split()) for t in texts]
    f[f'{text_col}_tokens'] = [len(tok.tokenize(t)) for t in texts]
    f[f'{text_col}_fertility'] = f[f'{text_col}_tokens'] / f[f'{text_col}_words']
    return f

def check_cuts(config_path):
    """The plan's cut-offs must match config_v4.json to four decimals."""
    c = json.load(open(config_path))
    ref = {'10%': c['threshold_10pct'], '20%': c['threshold_20pct'], '30%': c['threshold_30pct'],
           '50%': c['threshold'], '70%': c['threshold_70pct']}
    for k, v in ref.items():
        assert round(v, 4) == CUTS[k], f'cut-off {k}: config {v} vs plan {CUTS[k]}'
    return ref

# ---------- methods (all fitted on training rows only) ----------
class M0:                      # no correction
    name = 'M0 none'
    def fit(self, X, e): return self
    def predict(self, X): return X['hi_score'].to_numpy()

class M2:                      # quantile mapping, K knots, clipped at the ends
    def __init__(self, K=21): self.K = K; self.name = f'M2 quantile map K={K}'
    def fit(self, X, e):
        q = np.linspace(0, 1, self.K)
        self.hk = np.quantile(X['hi_score'], q); self.ek = np.quantile(e, q); return self
    def predict(self, X): return np.interp(X['hi_score'], self.hk, self.ek)

class M3:                      # linear regression on the allowed inputs
    name = 'M3 linear'
    def fit(self, X, e): self.m = LinearRegression().fit(X[FEATS], e); return self
    def predict(self, X): return np.clip(self.m.predict(X[FEATS]), 0, 1)

class M4:                      # small gradient boosting
    def __init__(self, depth=2, n=50): self.depth, self.n = depth, n; self.name = f'M4 boosting d={depth} n={n}'
    def fit(self, X, e):
        self.m = GradientBoostingRegressor(max_depth=self.depth, n_estimators=self.n, learning_rate=0.05,
                                           subsample=1.0, random_state=SEED).fit(X[FEATS], e); return self
    def predict(self, X): return np.clip(self.m.predict(X[FEATS]), 0, 1)

def share_matched_cutoffs(h_train, e_train, cuts=CUTS):
    """M1: Hinglish cut-off that sends the same share to the strong model as English does at the Arena cut-off."""
    return {k: float(np.quantile(h_train, 1 - np.mean(e_train >= t))) for k, t in cuts.items()}

# ---------- measures ----------
def sd_ratio(e, c): return float(np.std(e, ddof=1) / np.std(c, ddof=1))
def mae(e, c): return float(np.mean(np.abs(np.asarray(e) - np.asarray(c))))
def orth_slope(x, y):
    sxx, syy, sxy = np.var(x, ddof=1), np.var(y, ddof=1), np.cov(x, y)[0, 1]
    return float((syy - sxx + np.sqrt((syy - sxx) ** 2 + 4 * sxy ** 2)) / (2 * sxy))

def holm(p):
    p = np.asarray(p); order = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1, run)
    return adj

def routing_table(e, c, cutoff_for_c=None):
    """English decision uses the Arena cut-off. Corrected-Hinglish decision uses the same cut-off,
    or its own (M1) if cutoff_for_c is a dict of cut-offs."""
    rows = []
    for k, t in CUTS.items():
        de = np.asarray(e) >= t
        dc = np.asarray(c) >= (cutoff_for_c[k] if cutoff_for_c else t)
        b, cc = int(np.sum(de & ~dc)), int(np.sum(~de & dc))                  # English-only, Hinglish-only
        p = 1.0 if b + cc == 0 else stats.binomtest(min(b, cc), b + cc, 0.5).pvalue
        kappa = cohen_kappa_score(de, dc) if len(set(de)) > 1 and len(set(dc)) > 1 else float('nan')
        rows.append(dict(cut=k, en_strong=de.mean(), c_strong=dc.mean(), en_only=b, hi_only=cc,
                         agree=float(np.mean(de == dc)), kappa=kappa, p=p))
    t = pd.DataFrame(rows); t['p_holm'] = holm(t['p'].to_numpy()); return t

def auc_hard_easy(score, difficulty, easy=('Easy',), hard=('Hard', 'Very Hard')):
    d = np.asarray(difficulty); m = np.isin(d, easy + hard)
    y = np.isin(d[m], hard).astype(int)
    if len(set(y)) < 2: return float('nan')            # needs both Easy and Hard rows
    return float(roc_auc_score(y, np.asarray(score)[m]))

def bootstrap_ci(fn, n, B=10000, seed=SEED):
    rng = np.random.default_rng(seed); v = [fn(rng.integers(0, n, n)) for _ in range(B)]
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

# ---------- cross-validation (split by pair; one row per pair so twins never separate) ----------
def cv_predict(make_model, X, e, k=10):
    oof = np.zeros(len(e))
    for tr, te in KFold(k, shuffle=True, random_state=SEED).split(X):
        oof[te] = make_model().fit(X.iloc[tr], e[tr]).predict(X.iloc[te])
    return oof

def cv_share_matched(X, e, k=10):
    """Out-of-fold M1 decisions: one boolean array per cut-off."""
    out = {c: np.zeros(len(e), bool) for c in CUTS}
    for tr, te in KFold(k, shuffle=True, random_state=SEED).split(X):
        cut = share_matched_cutoffs(X['hi_score'].to_numpy()[tr], e[tr])
        for c in CUTS: out[c][te] = X['hi_score'].to_numpy()[te] >= cut[c]
    return out

def mean_agree(e, c, cutoff_for_c=None): return float(routing_table(e, c, cutoff_for_c)['agree'].mean())

def mae_se(e, pred):
    a = np.abs(e - pred); return float(a.std(ddof=1) / np.sqrt(len(a)))

def tune_and_select(X, e, k=10):
    """Plan section 5: tune on CV MAE (ties -> simpler), then apply the selection rule for the final model."""
    cands = {'M2': [lambda K=K: M2(K) for K in (11, 21, 51, 101)],
             'M3': [lambda: M3()],
             'M4': [lambda d=d, n=n: M4(d, n) for d in (1, 2, 3) for n in (25, 50, 100)]}
    best = {}
    for fam, makers in cands.items():
        res = []
        for mk in makers:
            oof = cv_predict(mk, X, e, k); res.append((mae(e, oof), mae_se(e, oof), mk, oof))
        res.sort(key=lambda r: r[0]); best[fam] = res[0]          # makers are listed simplest-first, sort is stable
    m1 = cv_share_matched(X, e, k)
    def agree_dec(dec): return float(np.mean([np.mean((e >= t) == dec[c]) for c, t in CUTS.items()]))
    agree_m1 = agree_dec(m1)
    agree = {f: mean_agree(e, best[f][3]) for f in best}
    final = 'M2'
    for fam in ('M3', 'M4'):
        if best['M2'][0] - best[fam][0] > best['M2'][1] and agree[fam] > agree_m1: final = fam
    return best, agree, agree_m1, final

## A2. Final-run functions (defined only, not executed here)

In [ ]:
import os
def _refuse_overwrite(path):
    if os.path.exists(path): raise FileExistsError(f'{path} exists. Never overwrite: save a new version name.')

def load_pairs(scores_path, feats_path, difficulty_col=None):
    """Returns X (allowed inputs), e (English score), and difficulty (or None).
    Expected columns, adapt the renames below if the third-set files name them differently:
      scores: pair_id, english_score (or english_bert), hinglish_score (or hinglish_bert)   features: pair_id, hinglish_words, hinglish_tokens, hinglish_fertility"""
    s = pd.read_csv(scores_path)
    s = s.rename(columns={'english_bert': 'english_score', 'hinglish_bert': 'hinglish_score'})   # Person02c layout (BERT columns only; _mf ignored)
    f = pd.read_csv(feats_path)
    f = f.rename(columns={'hi_words': 'hinglish_words', 'hi_tokens': 'hinglish_tokens', 'hi_fertility': 'hinglish_fertility'})
    d = s.merge(f, on='pair_id', validate='one_to_one')
    assert d.pair_id.is_unique and d[['english_score', 'hinglish_score']].notna().all().all(), 'bad ids or missing scores'
    assert d[['english_score', 'hinglish_score']].apply(lambda c: c.between(0, 1)).all().all(), 'score outside 0..1'
    X = pd.DataFrame({'pair_id': d.pair_id, 'hi_score': d.hinglish_score, 'hi_tokens': d.hinglish_tokens,
                      'hi_words': d.hinglish_words, 'hi_fertility': d.hinglish_fertility}).reset_index(drop=True)
    return X, d.english_score.to_numpy(), (d[difficulty_col].to_numpy() if difficulty_col else None)

def run_final(third, test, out_dir='.', B=10000):
    """third/test are (scores_path, feats_path) tuples. Fits on third, applies ONCE to test.
    Only call after the plan is signed (see the guard cell)."""
    Xt, et, _ = load_pairs(*third)
    best, agree, agree_m1, final = tune_and_select(Xt.drop(columns='pair_id'), et)
    maker = best[final][2]; model = maker().fit(Xt.drop(columns='pair_id'), et)
    cfg = dict(final_model=model.name, final_family=final,
               cv_mae={f: best[f][0] for f in best}, cv_mean_agree={**agree, 'M1': agree_m1},
               third_scores_sha256=sha256(third[0]), test_scores_sha256=sha256(test[0]), seed=SEED, cuts=CUTS)
    cfgp = os.path.join(out_dir, 'config_corr_v1.json'); _refuse_overwrite(cfgp); json.dump(cfg, open(cfgp, 'w'), indent=2)
    # ---- the single look at the test set ----
    Xe, ee, diff = load_pairs(*test, difficulty_col='difficulty')
    corr = model.predict(Xe.drop(columns='pair_id'))
    out = pd.DataFrame({'pair_id': Xe.pair_id, 'english_score': ee, 'hinglish_score': Xe.hi_score, 'hinglish_corrected': corr})
    outp = os.path.join(out_dir, 'corrected_scores_conf_v1.csv'); _refuse_overwrite(outp); out.to_csv(outp, index=False)
    h = Xe.hi_score.to_numpy(); n = len(ee); res = {}
    res['C1 SD ratio original'] = sd_ratio(ee, h)
    res['C1 SD ratio corrected'] = sd_ratio(ee, corr); res['C1 CI'] = bootstrap_ci(lambda i: sd_ratio(ee[i], corr[i]), n, B)
    res['orth slope original / corrected'] = (orth_slope(ee, h), orth_slope(ee, corr))
    res['MAE original / corrected'] = (mae(ee, h), mae(ee, corr))
    res['MAE improvement CI'] = bootstrap_ci(lambda i: mae(ee[i], h[i]) - mae(ee[i], corr[i]), n, B)
    tab0, tab1 = routing_table(ee, h), routing_table(ee, corr)
    res['C2 any Holm p < 0.05 after correction'] = bool((tab1.p_holm < 0.05).any())
    a0, a1 = auc_hard_easy(h, diff, easy=('Easy',), hard=('Hard',)), auc_hard_easy(corr, diff, easy=('Easy',), hard=('Hard',))
    res['C3 AUC original / corrected'] = (a0, a1)
    res['C3 AUC change CI'] = bootstrap_ci(lambda i: auc_hard_easy(corr[i], diff[i], ('Easy',), ('Hard',)) - auc_hard_easy(h[i], diff[i], ('Easy',), ('Hard',)), n, B)
    if final != 'M2':      # B1 only applies when a learned model is final
        cuts_m1 = share_matched_cutoffs(Xt.hi_score.to_numpy(), et)
        res['B1 mean agreement final / M1'] = (mean_agree(ee, corr), mean_agree(ee, h, cuts_m1))
    tab0.to_csv(os.path.join(out_dir, 'correction_routing_original_v1.csv'), index=False) if not os.path.exists(os.path.join(out_dir, 'correction_routing_original_v1.csv')) else None
    tab1.to_csv(os.path.join(out_dir, 'correction_routing_corrected_v1.csv'), index=False) if not os.path.exists(os.path.join(out_dir, 'correction_routing_corrected_v1.csv')) else None
    pd.Series({k: str(v) for k, v in res.items()}).to_csv(os.path.join(out_dir, 'correction_results_v1.csv'), header=['value'])
    return res, tab0, tab1

## B. Dry run on the 80 pilot pairs (test of the code, not a result)

In [ ]:
# DRY RUN on the 80 PILOT pairs only. The 152 confirmatory scores are never loaded in this section.
import warnings; warnings.filterwarnings('ignore')
s = pd.read_csv('scores_v2.csv'); f = pd.read_csv('features_v2.csv')
p = pd.read_excel('benchmark_v2.xlsx', sheet_name='Benchmark')
assert s.pair_id.tolist() == f.pair_id.tolist() == p.pair_id.tolist() and s.pair_id.is_unique and s.notna().all().all()
print('cut-offs match config_v4.json:', check_cuts('config_v4.json'))

X = pd.DataFrame({'hi_score': s.hinglish_score, 'hi_tokens': f.hi_tokens, 'hi_words': f.hi_words, 'hi_fertility': f.hi_fertility})
e = s.english_score.to_numpy(); diff = p.difficulty.to_numpy()

best, agree, agree_m1, final = tune_and_select(X, e)
for fam, (m, se, mk, oof) in best.items():
    print(fam, mk().name, '| CV MAE', round(m, 4), '+/-', round(se, 4), '| mean routing agreement', round(agree[fam], 3))
print('M1 share-matched agreement', round(agree_m1, 3), '| M0 none', round(mean_agree(e, X.hi_score.to_numpy()), 3), '| final model by the rule:', final)

In [ ]:
oof = best['M2'][3]
print('SD ratio  original -> M2:', round(sd_ratio(e, X.hi_score), 2), '->', round(sd_ratio(e, oof), 2))
print('MAE       original -> M2:', round(mae(e, X.hi_score), 4), '->', round(mae(e, oof), 4))
print('Orth slope original -> M2:', round(orth_slope(e, X.hi_score), 2), '->', round(orth_slope(e, oof), 2))
print('\nRouting, original Hinglish:'); display(routing_table(e, X.hi_score.to_numpy()).round(3))
print('Routing, M2 corrected (out-of-fold):'); display(routing_table(e, oof).round(3))
m1 = cv_share_matched(X, e); d2 = {c: (oof >= t) for c, t in CUTS.items()}
print('M1 and M2 send the same pairs to the strong model:', {c: round(float(np.mean(m1[c] == d2[c])), 3) for c in CUTS})
print('AUC Easy vs Hard/Very Hard: English', round(auc_hard_easy(e, diff), 3), '| Hinglish', round(auc_hard_easy(X.hi_score, diff), 3), '| M2', round(auc_hard_easy(oof, diff), 3))

## B2. Compute the third-set Hinglish features (needs only the text, not the scores)
Person02c does not save features, so we compute them ourselves with the router's own tokenizer. Run this once the merged third-set file (all 200 rows OK) exists. Does not touch any test data.


In [ ]:
# from transformers import AutoTokenizer
# tok = AutoTokenizer.from_pretrained('routellm/bert_gpt4_augmented')      # same tokenizer as Person02
# third = pd.read_csv('thirdset_merged.csv')                               # TODO: merged third-set file from Aryan (needs pair_id, hinglish)
# feats = compute_features(third, tok)
# assert feats.pair_id.is_unique and (feats.hinglish_words > 0).all()
# feats.to_csv('features_thirdset_v1.csv', index=False)                    # new name, never overwrite


## C. Final run (locked until the plan is signed)

In [ ]:
# FINAL RUN (do NOT run until all four people have signed correction_model_plan)
PLAN_SIGNED = False          # change to True only after sign-off
if not PLAN_SIGNED:
    raise SystemExit('Plan not signed yet. Stop here.')

THIRD = ('scores_thirdset_v1.csv', 'features_thirdset_v1.csv')       # scores from Reshmi (Person02c); features from section B2
TEST  = ('scores_confirmatory_v1.csv', 'features_confirmatory_v1.csv')
# Before running: check the hashes against HASHES.md and the config_thirdset_v1.json that Reshmi sends.
# res, tab_orig, tab_corr = run_final(THIRD, TEST, out_dir='.')       # runs ONCE; refuses to overwrite files